# G6V1 - Integración de datos desde CSV, Excel, JSON y SQL relacional

**Versión aprendiz.**

En este notebook se integran cuatro fuentes estructuradas para construir un dataset único por cliente.

## 1. Propósito

Construir un dataset integrado usando `ID_Cliente` como llave común. La integración combina datos maestros, transacciones de compras, interacciones web y satisfacción del cliente.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../03_data")
OUT_DIR = Path("../08_revision/evidencias")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())

## 2. Carga de fuentes

Se cargan archivos en cuatro formatos: CSV, Excel, JSON y SQL relacional.

In [ ]:
clientes = pd.read_csv(DATA_DIR / "clientes.csv")

try:
    compras = pd.read_excel(DATA_DIR / "compras.xlsx", sheet_name="compras")
except Exception as e:
    print("No se pudo leer compras.xlsx. Se usará compras_respaldo.csv. Detalle:", e)
    compras = pd.read_csv(DATA_DIR / "compras_respaldo.csv")

web = pd.read_json(DATA_DIR / "interacciones_web.json")

conn = sqlite3.connect(DATA_DIR / "empresa_integracion.db")
satisfaccion = pd.read_sql_query("SELECT * FROM satisfaccion_clientes", conn)
conn.close()

print("clientes:", clientes.shape)
print("compras:", compras.shape)
print("web:", web.shape)
print("satisfaccion:", satisfaccion.shape)

## 3. Diagnóstico inicial

Antes de unir, se revisa la calidad de cada fuente.

In [ ]:
fuentes = {
    "clientes": clientes,
    "compras": compras,
    "web": web,
    "satisfaccion": satisfaccion
}

for nombre, df in fuentes.items():
    print("\nFuente:", nombre)
    print("Dimensiones:", df.shape)
    print("Duplicados:", df.duplicated().sum())
    print("Nulos por columna:")
    print(df.isna().sum())

### Análisis

Responda: ¿cuál fuente tiene mayor granularidad? ¿Cuál fuente es maestro de datos? ¿Qué problemas de calidad se observan?

## 4. Normalización de llaves y fechas

La llave `ID_Cliente` debe quedar limpia y comparable en todas las fuentes.

In [ ]:
for df in [clientes, compras, web, satisfaccion]:
    df["ID_Cliente"] = df["ID_Cliente"].astype(str).str.strip().str.upper()

clientes["FechaRegistro"] = pd.to_datetime(clientes["FechaRegistro"], errors="coerce")
compras["FechaCompra"] = pd.to_datetime(compras["FechaCompra"], errors="coerce")

clientes.head()

## 5. Validación de correspondencia

Se identifican transacciones que no tienen cliente asociado. Esto es importante para la trazabilidad.

In [ ]:
ids_clientes = set(clientes["ID_Cliente"])
compras_sin_cliente = compras[~compras["ID_Cliente"].isin(ids_clientes)]

print("Compras con ID_Cliente inexistente en clientes:", len(compras_sin_cliente))
compras_sin_cliente.head()

## 6. Agregación de compras por cliente

Como compras tiene varias filas por cliente, primero se agregan las transacciones para evitar duplicar clientes en el dataset final.

In [ ]:
compras["ValorNeto"] = compras["Cantidad"] * compras["ValorUnitario"] * (1 - compras["Descuento"])
compras_validas = compras[compras["ID_Cliente"].isin(ids_clientes)].copy()

compras_agregadas = (
    compras_validas
    .groupby("ID_Cliente")
    .agg(
        TotalCompras=("ID_Cliente", "count"),
        ValorTotalCompras=("ValorNeto", "sum"),
        TicketPromedio=("ValorNeto", "mean"),
        UltimaCompra=("FechaCompra", "max")
    )
    .reset_index()
)

compras_agregadas["ValorTotalCompras"] = compras_agregadas["ValorTotalCompras"].round(2)
compras_agregadas["TicketPromedio"] = compras_agregadas["TicketPromedio"].round(2)

compras_agregadas.head()

## 7. Integración de fuentes

Se usa `merge` tipo `left` para conservar todos los clientes del maestro.

In [ ]:
dataset = clientes.merge(compras_agregadas, on="ID_Cliente", how="left")
dataset = dataset.merge(web, on="ID_Cliente", how="left")
dataset = dataset.merge(satisfaccion, on="ID_Cliente", how="left")

dataset["TotalCompras"] = dataset["TotalCompras"].fillna(0).astype(int)
dataset["ValorTotalCompras"] = dataset["ValorTotalCompras"].fillna(0)
dataset["TicketPromedio"] = dataset["TicketPromedio"].fillna(0)

print("Dataset integrado:", dataset.shape)
dataset.head()

## 8. Validación del dataset integrado

In [ ]:
print("Duplicados por ID_Cliente:", dataset["ID_Cliente"].duplicated().sum())
print("\nNulos finales:")
print(dataset.isna().sum())
print("\nFilas y columnas:", dataset.shape)

## 9. Indicadores básicos

In [ ]:
indicadores_segmento = (
    dataset.groupby("Segmento")
    .agg(
        Clientes=("ID_Cliente", "count"),
        ValorTotal=("ValorTotalCompras", "sum"),
        TicketPromedio=("TicketPromedio", "mean"),
        QuejasPromedio=("QuejasUltimos6M", "mean")
    )
    .round(2)
    .reset_index()
)

indicadores_ciudad = (
    dataset.groupby("Ciudad")
    .agg(
        Clientes=("ID_Cliente", "count"),
        ValorTotal=("ValorTotalCompras", "sum"),
        VisitasPromedio=("VisitasWebUltimoMes", "mean")
    )
    .round(2)
    .reset_index()
)

display(indicadores_segmento)
display(indicadores_ciudad)

## 10. Visualización

In [ ]:
plt.figure(figsize=(8, 5))
indicadores_segmento.plot(kind="bar", x="Segmento", y="ValorTotal", legend=False)
plt.title("Valor total de compras por segmento")
plt.xlabel("Segmento")
plt.ylabel("Valor total")
plt.tight_layout()
plt.savefig(OUT_DIR / "valor_total_por_segmento.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
indicadores_ciudad.plot(kind="bar", x="Ciudad", y="ValorTotal", legend=False)
plt.title("Valor total de compras por ciudad")
plt.xlabel("Ciudad")
plt.ylabel("Valor total")
plt.tight_layout()
plt.savefig(OUT_DIR / "valor_total_por_ciudad.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
dataset["Satisfaccion"].value_counts().plot(kind="bar")
plt.title("Distribución de satisfacción")
plt.xlabel("Satisfacción")
plt.ylabel("Cantidad de clientes")
plt.tight_layout()
plt.savefig(OUT_DIR / "distribucion_satisfaccion.png", dpi=150)
plt.show()

## 11. Exportación de resultados

In [ ]:
dataset.to_csv(OUT_DIR / "dataset_integrado_clientes.csv", index=False, encoding="utf-8-sig")
indicadores_segmento.to_csv(OUT_DIR / "indicadores_segmento.csv", index=False, encoding="utf-8-sig")
indicadores_ciudad.to_csv(OUT_DIR / "indicadores_ciudad.csv", index=False, encoding="utf-8-sig")

reporte = []
reporte.append("REPORTE DE VALIDACIÓN G6V1")
reporte.append("=" * 40)
reporte.append(f"Clientes fuente: {len(clientes)}")
reporte.append(f"Compras fuente: {len(compras)}")
reporte.append(f"Compras sin cliente: {len(compras_sin_cliente)}")
reporte.append(f"Filas dataset integrado: {len(dataset)}")
reporte.append(f"Columnas dataset integrado: {dataset.shape[1]}")
reporte.append(f"Duplicados por ID_Cliente en dataset final: {dataset['ID_Cliente'].duplicated().sum()}")

with open(OUT_DIR / "reporte_validacion_integracion.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(reporte))

print("Archivos exportados en:", OUT_DIR)

## 12. Preguntas de análisis

1. ¿Por qué se agregaron las compras antes de integrarlas?
2. ¿Qué significa que existan compras con ID_Cliente inexistente?
3. ¿Qué variables del dataset integrado podrían servir para análisis comercial?
4. ¿Qué limitaciones tendría este dataset si se usara para IA?
5. ¿Qué controles de calidad agregaría en una empresa real?